# Ijeoma AI — Train and Evaluate

Companion to **Lab 8.9**, wired to the Ijeoma dataset and the grounded task
design: every prompt carries its passage, so the model reads rather than
recalls.

Three things this notebook does that the lab template does not:

1. **Scores the untuned model first.** Same prompts, same test set, same
   metrics. Without that number you cannot say fine-tuning helped.
2. **Scores per task.** `classify`, `answer`, `summarize` and `refuse` fail in
   different ways; one averaged number hides all of it.
3. **Measures the invention rate.** Prompts about places that are not in the
   corpus, where the only correct answer is to decline.

---

### Run the cells in order, top to bottom

Two rules, both of which cost a run when broken:

1. **Cell 1 must run before anything imports Keras.** The backend is fixed at
   import time. If you have already imported Keras, go to
   **Runtime → Restart session** and start again from cell 1.
2. **Runtime → Change runtime type → T4 GPU** before you start.

`SMOKE_TEST = True` is the default: 200 examples, 1 epoch, end to end in
roughly fifteen minutes. Get a clean pass, then set it to `False`.

---

### What changed in this revision

Read this if you ran the earlier version — the numbers it produced were not
measuring what they claimed to.

- **The token secret is `IjeomaHF`**, and the loader tries a list of names
  rather than one hard-coded `HF_TOKEN`. The `hf://` presets are now first in
  the candidate list, since this account has a Hugging Face token and no
  Kaggle one.
- **The `transformers` / `peft` / `trl` cell is gone.** It downloaded a second
  copy of Gemma as a Torch model, bound it to the name `model`, and left
  roughly 2 GB sitting on the T4 — which the next cell then overwrote with the
  Keras model. Nothing in the notebook fine-tuned that Torch copy; it only
  took memory away from the run that mattered.
- **Decoding is pinned to greedy.** KerasHub compiles a `top_k` sampler in by
  default, so the old baseline and tuned scores were both sampled, and
  "did JSON parse" was partly a coin flip.
- **`<end_of_turn>` is now a stop token.** Without it every generation ran to
  `max_length` — most of the ten-to-twenty seconds per item — and trailed off
  into an invented second turn.
- **The training and inference templates match.** Training wrote
  `…<end_of_turn><start_of_turn>model`, inference sent
  `…<end_of_turn>\n<start_of_turn>model`. One newline, scored for free.
- **The cached generate function is dropped after `enable_lora` and after
  every `fit`.** It is compiled once and reused, so without the reset the
  "tuned" evaluation can be the untuned model answering.
- **A real optimizer.** Explicit `compile` with AdamW, weight decay kept off
  bias and scale weights, and gradient accumulation so the effective batch is
  4 rather than 1.
- **A validation slice**, held out of the fit set, so you see the epoch where
  it starts overfitting.
- **The training subset is shuffled** before it is cut to `TRAIN_SUBSET`;
  it used to take the first 200 rows in file order.
- **Evaluation generates in batches**, which is most of the wall clock.
- **The adapter is saved on its own** (a few MB) instead of writing all
  3.7 GB of base weights to Drive on every run.

## 1. Setup — run this first

In [ ]:
# Keras reads KERAS_BACKEND when it is first imported, so this must happen
# before any import of keras, keras_hub or keras_nlp. If Keras is already in
# sys.modules the backend is locked in and only a restart clears it.
import sys

if "keras" in sys.modules:
    raise RuntimeError(
        "Keras is already imported, so the backend is locked. "
        "Runtime -> Restart session, then run this cell first."
    )

import os

os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
print("backend requested: jax")

In [ ]:
# keras-nlp is the old name for keras-hub and nothing here imports it; having
# both installed is a good way to get a version pin fight. Install what is used.
!pip install -q -U keras keras-hub

In [ ]:
import json
import random
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
import keras_hub as kh

print("keras      ", keras.__version__)
print("keras_hub  ", kh.__version__)
print("backend    ", keras.backend.backend())

if keras.backend.backend() != "jax":
    print("\nWARNING: backend is not jax. Something imported keras before "
          "cell 1 ran. Runtime -> Restart session and start again from the top.")

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU - Runtime > Change runtime type > T4"

### Credentials

Gemma weights live behind a licence gate. You need **one** of these working;
the loader tries them in order and tells you which one succeeded.

**Hugging Face (what this project uses)**

1. Accept the licence on
   [google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it).
2. Settings → Access Tokens → create a **read** token.
3. In Colab, 🔑 in the left sidebar → add a secret named **`IjeomaHF`**, value
   pasted **without quotes**, **Notebook access ON**.

The name matters: a Colab secret is only visible under the exact name you gave
it. `HF_SECRET_NAMES` in the config cell is the list the loader tries, so if
you called yours something else, add it there rather than renaming the secret.

**Kaggle (the route the course notes use)**

1. Accept the licence on the [Gemma 3 Keras model
   page](https://www.kaggle.com/models/keras/gemma3/keras/) — *Request
   Access*, read, accept. Nothing below works until this is done, and a
   missing acceptance looks exactly like a wrong key: HTTP 403.
2. Kaggle → profile icon → **Settings** → **API** → **Create New Token**,
   which downloads `kaggle.json` containing a username and a key.
3. Add both as Colab secrets, `KAGGLE_USERNAME` and `KAGGLE_KEY`, notebook
   access on.

> **Two failures worth recognising.** A bare preset name like
> `gemma3_instruct_1b` always resolves to Kaggle, whatever token you loaded —
> so loading an HF token and asking for a bare preset sends an unauthenticated
> request to Kaggle and returns 403. And a 401 at login means the token is
> wrong or revoked, while a 403 during the weights download means the licence
> has not been accepted on that account. They are different problems.

In [ ]:
from google.colab import userdata, drive
import os


def get_secret(name):
    try:
        v = userdata.get(name)
        return v.strip() if v else None
    except Exception:
        return None


# Colab secret names, tried in order. "IjeomaHF" is this project's; the rest
# are here so the notebook still runs on someone else's account.
HF_SECRET_NAMES = ["IjeomaHF", "HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACEHUB_API_TOKEN"]
KAGGLE_SECRET_PAIRS = [("KAGGLE_USERNAME", "KAGGLE_KEY")]

HAVE_KAGGLE = False
for user_key, api_key in KAGGLE_SECRET_PAIRS:
    u, k = get_secret(user_key), get_secret(api_key)
    if u and k:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = u, k
        HAVE_KAGGLE = True
        print(f"Kaggle credentials loaded from {user_key}/{api_key} (user: {u})")
        break
if not HAVE_KAGGLE:
    print("Kaggle credentials not found - the Kaggle presets will be skipped")

HAVE_HF, HF_SECRET_USED = False, None
for name in HF_SECRET_NAMES:
    tok = get_secret(name)
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
        HAVE_HF, HF_SECRET_USED = True, name
        print(f"Hugging Face token loaded from secret '{name}'")
        break
if not HAVE_HF:
    print("Hugging Face token NOT found. Add it as a Colab secret named one of: "
          + ", ".join(HF_SECRET_NAMES))

# keras-hub reads HF_TOKEN from the environment, so this login is not strictly
# required - it is here to fail now, with a readable message, rather than as a
# 401 halfway through a 2 GB download.
if HAVE_HF:
    try:
        from huggingface_hub import login, whoami
        login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
        print("  authenticated with Hugging Face as", whoami().get("name"))
    except Exception as exc:
        print("  WARNING: the token loaded but login failed:", str(exc)[:200])
        print("  401 here = the token is wrong or revoked. Regenerate it.")

if not (HAVE_KAGGLE or HAVE_HF):
    print("\nNeither credential is set. Add one in the Secrets panel (key icon, "
          "left sidebar), switch Notebook access on, and re-run this cell.")

try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount status:", e)

## 2. Config

Every knob in one place, so the experiment log at the bottom records what you
actually ran.

**On the preset.** The lab template loads `gemma3_1b`, the pretrained base
model, and argues it is a more neutral starting point for LoRA. Fair, but it
makes the baseline comparison empty: a base model has never been trained to
follow an instruction, so of course it rambles, and "fine-tuning beat it"
tells you nothing. Ijeoma's tasks are reading tasks and instruction-following
is most of the job, so the instruction-tuned checkpoint is the default.

**On the candidate order.** A *bare* preset name resolves to Kaggle and
nothing else, so `gemma3_instruct_1b` with only an HF token in the environment
is a guaranteed 403. The `hf://` URIs go first, and the bare name stays last
as a fallback for whoever runs this with Kaggle credentials instead.

Note there is no `_en` suffix on Gemma 3 presets — `gemma3_instruct_1b_en`
does not exist, and the error it produces ("no config.json") looks like a
permissions problem rather than a typo.

In [ ]:
# ---- data -----------------------------------------------------------------
# Project folder "Ijeoma" (Usman's My Drive). Anyone else running this needs a
# shortcut to it in their own My Drive: right-click the folder > Organize >
# Add shortcut > My Drive. Found automatically up to two levels deep.
import glob
_hits = [d for d in ["/content/drive/MyDrive/Ijeoma"]
         + glob.glob("/content/drive/MyDrive/*/Ijeoma")
         + glob.glob("/content/drive/MyDrive/*/*/Ijeoma")
         if os.path.exists(f"{d}/conversational_training_data.jsonl")]
if not _hits:
    raise FileNotFoundError(
        "Could not find the 'Ijeoma' folder with the training data under "
        "/content/drive/MyDrive. Mount Drive (section 2), or add a shortcut "
        "to the shared folder in your My Drive.")
DRIVE = _hits[0]
print("project folder:", DRIVE)
TRAIN_PATH = f"{DRIVE}/conversational_training_data.jsonl"
TEST_PATH = f"{DRIVE}/conversational_test_data.jsonl"

# The _full files carry task/subtype/country/passage_id. Without them the
# notebook falls back to reading the task from each prompt's tag, which works
# but gives up the per-country breakdown and the leakage check.
TRAIN_FULL_PATH = f"{DRIVE}/pairs_train_full.jsonl"
TEST_FULL_PATH = f"{DRIVE}/pairs_test_full.jsonl"

# "instruction" = classify/answer/refuse, "summarization" = the extractive
# summarization set, "both" = concatenated. The two sets share one
# passage-level split, so combining them does not leak. If the summarization
# files are not on Drive this quietly becomes "instruction".
DATASET = "both"
SUM_TRAIN_PATH = f"{DRIVE}/summarization_train_full.jsonl"
SUM_TEST_PATH = f"{DRIVE}/summarization_test_full.jsonl"

# ---- model ----------------------------------------------------------------
# Tried in order; the first that loads wins. A bare name resolves to Kaggle
# regardless of which token is in the environment, so it goes last.
PRESET_CANDIDATES = [
    "hf://google/gemma-3-1b-it",       # Hugging Face (needs HF token + licence)
    "hf://keras/gemma3_instruct_1b",   # Keras mirror on Hugging Face
    "gemma3_instruct_1b",              # Kaggle (needs KAGGLE_* + licence)
]

# ---- run size -------------------------------------------------------------
SMOKE_TEST = False
# Training cap. ~1.3 s per example on a T4. None = the full training set
# (~2 h per epoch). Set e.g. 1500 (~35 min) if the session is short on time.
TRAIN_SUBSET = 200 if SMOKE_TEST else None
CHECKPOINT_EVERY = 500      # steps; LoRA weights saved to Drive this often
VAL_N = 32                  # held out of the training split, for epoch loss
EVAL_N = 20 if SMOKE_TEST else 100   # test items scored per model
ADVERSARIAL_N = 20
GEN_BATCH_SIZE = 4          # generations per forward pass during evaluation

# ---- LoRA -----------------------------------------------------------------
LORA_RANK = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 1
BATCH_SIZE = 1              # a T4 at this sequence length fits one example
GRAD_ACCUM_STEPS = 4        # effective batch = BATCH_SIZE * GRAD_ACCUM_STEPS
MAX_SEQUENCE_LENGTH = 1024  # ceiling for the measurement in section 5
SEQUENCE_LENGTH = None      # measured from the data in section 5
GEN_MAX_LENGTH = None       # derived from SEQUENCE_LENGTH

# ---- extra metrics (section 10b) ------------------------------------------
# LLM-as-judge uses OpenAI. Add OPENAI_API_KEY to Colab Secrets and enable
# notebook access; without it the judge is skipped and everything else runs.
JUDGE_MODEL = "gpt-5.4-mini"
JUDGE_N = 40                # grounded items judged per model

# ---- saving ---------------------------------------------------------------
# The adapter is a few MB. The full checkpoint is 3.7 GB of base weights you
# already have, written to Drive on every run, so it is off by default.
SAVE_FULL_WEIGHTS = False

# ---- misc -----------------------------------------------------------------
SEED = 812
MIXED_PRECISION = False
PRESET = None               # set by the loader

keras.utils.set_random_seed(SEED)
random.seed(SEED)
print("smoke test:", SMOKE_TEST, "| dataset:", DATASET,
      "| effective batch:", BATCH_SIZE * max(1, GRAD_ACCUM_STEPS))

**On mixed precision.** It roughly halves memory, and the T4 is a Turing card
with no native bfloat16 — so if you enable it, it must be `mixed_float16`,
never `mixed_bfloat16`. Off by default because float16 can produce NaN losses
with LoRA on JAX, and a silent NaN is a bad way to spend an evening.

In [ ]:
if MIXED_PRECISION:
    keras.mixed_precision.set_global_policy("mixed_float16")
    print("mixed precision ON - watch for NaN loss")
else:
    print("mixed precision off (float32)")

## 3. Load the data

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return pd.DataFrame([json.loads(l) for l in fh if l.strip()])


def infer_task(prompt):
    '''Read the task off the prompt tag.

    Lets per-task scoring work even when the _full metadata files are not on
    Drive - the tag is written into every input by the data builder.
    '''
    head = str(prompt)[:40].upper()
    if head.startswith("[CLASSIFY]"):
        return "classify"
    if head.startswith("[SUMMARIZE]"):
        return "summarize"
    if head.startswith("[DIGEST]"):
        return "digest"
    if head.startswith("[ANSWER]"):
        # Refusals arrive as ordinary questions; the gold output is what
        # distinguishes them, and only the _full files know for sure.
        return "answer"
    return "unknown"


for label, path in (("train", TRAIN_PATH), ("test", TEST_PATH),
                    ("train_full", TRAIN_FULL_PATH), ("test_full", TEST_FULL_PATH),
                    ("sum_train", SUM_TRAIN_PATH), ("sum_test", SUM_TEST_PATH)):
    print(f"{'found  ' if os.path.exists(path) else 'MISSING'}  {label:11s} {os.path.basename(path)}")

In [ ]:
train_df = load_jsonl(TRAIN_PATH)
test_df = load_jsonl(TEST_PATH)

HAVE_META = os.path.exists(TRAIN_FULL_PATH) and os.path.exists(TEST_FULL_PATH)
if HAVE_META:
    train_full, test_full = load_jsonl(TRAIN_FULL_PATH), load_jsonl(TEST_FULL_PATH)
else:
    print("_full files not on Drive - falling back to the prompt tag for task "
          "labels. Per-task scoring still works; the per-country breakdown and "
          "the leakage assertion do not. Upload pairs_train_full.jsonl and "
          "pairs_test_full.jsonl from the dataset bundle to get them back.\n")
    train_full, test_full = train_df.copy(), test_df.copy()
    for d in (train_full, test_full):
        d["task"] = d["input"].map(infer_task)
        d["subtype"] = None
        d["country"] = None
        d["passage_id"] = None

# Merge the summarization set when asked. Both were split on the same passage
# ids, so concatenating cannot put one passage on both sides.
if DATASET in ("summarization", "both"):
    if os.path.exists(SUM_TRAIN_PATH) and os.path.exists(SUM_TEST_PATH):
        s_tr, s_te = load_jsonl(SUM_TRAIN_PATH), load_jsonl(SUM_TEST_PATH)
        if DATASET == "summarization":
            train_full, test_full = s_tr, s_te
        else:
            train_full = pd.concat([train_full, s_tr], ignore_index=True)
            test_full = pd.concat([test_full, s_te], ignore_index=True)
        print(f"summarization merged: +{len(s_tr)} train, +{len(s_te)} test")
    else:
        print("summarization _full files not found - continuing without them")

train_data = train_full[["input", "output"]].to_dict("records")
test_data = test_full[["input", "output"]].to_dict("records")
print(f"\ntrain {len(train_data)}   test {len(test_data)}   metadata {HAVE_META}")
print("\ntest set by task:")
print(test_full["task"].value_counts().to_string())

In [ ]:
display(train_df.head(5))

In [ ]:
# Leakage check. Only meaningful with the _full files, which carry passage_id.
if HAVE_META and test_full["passage_id"].notna().any():
    tr = set(train_full["passage_id"].dropna())
    te = set(test_full["passage_id"].dropna())
    overlap = tr & te
    print(f"train passages {len(tr)} | test passages {len(te)} | overlap {len(overlap)}")
    assert not overlap, f"LEAKAGE: {len(overlap)} passages in both splits"
    print("no leakage")
else:
    print("skipped - needs the _full files")

## 4. Load Gemma

Loaded before the sequence-length measurement so the tokenizer comes from the
model itself — one download, one failure point, and no chance of the
tokenizer and the model disagreeing.

The `compile(sampler="greedy")` at the end matters more than it looks.
KerasHub compiles a sampler into a `CausalLM` for you, and the default is
`top_k` — random. Score a baseline under a sampler and score the tuned model
under the same sampler and a chunk of the difference between them is noise;
worse, "did the classifier emit valid JSON" becomes partly luck. Greedy
decoding is reproducible, which is the whole point of a before-and-after
table.

The previous revision had a cell here that `pip install`ed `transformers`,
`peft` and `trl`, downloaded Gemma a second time as a Torch model and assigned
it to `model` — which the loader below then overwrote. It fine-tuned nothing;
it just held about 2 GB of the T4 for the rest of the session. It is gone.

In [ ]:
def load_gemma(candidates):
    """Try each preset in turn; return the first that loads."""
    errors = []
    for preset in candidates:
        if preset.startswith("hf://") and not HAVE_HF:
            errors.append((preset, "skipped - no Hugging Face token"))
            continue
        if not preset.startswith("hf://") and not HAVE_KAGGLE:
            errors.append((preset, "skipped - no Kaggle credentials"))
            continue
        try:
            print(f"trying {preset} ...")
            t0 = time.time()
            m = kh.models.Gemma3CausalLM.from_preset(preset)
            print(f"  loaded in {time.time() - t0:.0f}s")
            return m, preset
        except Exception as exc:
            msg = str(exc).split("\n")[0][:160]
            errors.append((preset, msg))
            print(f"  failed: {msg}")
    print("\nNone of the presets loaded:\n")
    for p, e in errors:
        print(f"  {p}\n     {e}")
    print("\n403 almost always means the Gemma licence has not been accepted "
          "on the account the token belongs to, not that the key is wrong.")
    raise RuntimeError("could not load any Gemma preset - see the list above")


model, PRESET = load_gemma(PRESET_CANDIDATES)

# Pin the decoding strategy. Everything below - baseline, probe, tuned scores,
# invention rate - is then measuring the model rather than the sampler.
model.compile(sampler="greedy")
model.generate_function = None

short_preset = PRESET.split("/")[-1]
RUN_NAME = f"{short_preset}-r{LORA_RANK}-lr{LEARNING_RATE}-e{NUM_EPOCHS}"
print("\npreset:", PRESET)
print("run   :", RUN_NAME)
print("sampler: greedy")

model.summary()

## 5. Measure the sequence length

The lab template sets `sequence_length = 450`. Ijeoma's prompts carry a whole
passage, so 450 would cut most of them in half — and a truncated passage is
worse than no passage: the model is trained to answer a question whose
evidence has been deleted, which is a direct lesson in making things up.

The chat template is also defined here rather than further down, because the
same two functions have to build the training strings *and* the inference
prompt. When they drift apart — and in the previous revision they had, by one
newline — you train on one format and generate with another, and pay for it
in every metric at once.

In [ ]:
tokenizer = model.preprocessor.tokenizer
SOT, EOT = "<start_of_turn>", "<end_of_turn>"


def format_io_as_turns(input_text, output_text):
    """Wrap one pair as a user turn and a model turn.

    The prompt half ends with EOT *and a newline* because that is exactly what
    build_prompt() sends at inference time. Gemma's own template puts a
    newline there; training without it and generating with it is a free loss
    of accuracy.
    """
    return (f"{SOT}user\n{input_text}{EOT}\n",
            f"{SOT}model\n{output_text}{EOT}")


def build_prompt(question):
    """The inference-time string: the user turn plus an open model turn."""
    q, _ = format_io_as_turns(question, "")
    return q + f"{SOT}model\n"


def full_text(rec):
    q, a = format_io_as_turns(rec["input"], rec["output"])
    return q + a


sample = random.sample(train_data, min(400, len(train_data)))
lengths = np.array([len(tokenizer.tokenize(full_text(r))) for r in sample])
p50, p95, p99, mx = (int(np.percentile(lengths, q)) for q in (50, 95, 99, 100))
print(f"tokens  median {p50}  p95 {p95}  p99 {p99}  max {mx}")

if SEQUENCE_LENGTH is None:
    SEQUENCE_LENGTH = int(min(MAX_SEQUENCE_LENGTH,
                              64 * np.ceil((mx + 16) / 64)))
if GEN_MAX_LENGTH is None:
    # Room for the prompt plus an answer. Too small and long prompts leave no
    # tokens to answer in, which looks like a broken model.
    GEN_MAX_LENGTH = int(min(1536, SEQUENCE_LENGTH + 256))

model.preprocessor.sequence_length = SEQUENCE_LENGTH
truncated = float((lengths > SEQUENCE_LENGTH).mean())
print(f"\nSEQUENCE_LENGTH = {SEQUENCE_LENGTH}   GEN_MAX_LENGTH = {GEN_MAX_LENGTH}")
print(f"{truncated:.1%} of examples truncated")
if truncated > 0.01:
    print("WARNING: raise MAX_SEQUENCE_LENGTH, or shorten passages when "
          "building the data. If the T4 runs out of memory at this length, "
          "shorten the passages or turn on MIXED_PRECISION - do not silently "
          "truncate here.")

# Check the template round-trips through the tokenizer before 20 minutes of
# training depend on it.
_demo = build_prompt("[ANSWER] ping")
assert _demo.endswith(f"{SOT}model\n"), "inference prompt must end mid-turn"
assert format_io_as_turns("a", "b")[0] + f"{SOT}model\n" == build_prompt("a")
print("\ntemplate check ok. inference prompt looks like:\n", repr(_demo))

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(lengths, bins=40, color="#6b7280")
plt.axvline(SEQUENCE_LENGTH, color="#2563eb", linestyle="--",
            label=f"sequence_length = {SEQUENCE_LENGTH}")
plt.xlabel("tokens per training example")
plt.ylabel("examples")
plt.title("Nothing should sit to the right of the line")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Format as turns

Worth knowing what the dict does: the Gemma preprocessor takes
`{"prompts": ..., "responses": ...}` and returns `(x, y, sample_weight)` with
`sample_weight` 0 on prompt tokens and 1 on response tokens — so **the loss is
already computed on the response only**. You do not need to mask it yourself,
and you should not pass one concatenated string, which would train the model
to generate the passage back at you.

Two things happen here that did not before. The training pool is **shuffled
before it is cut** to `TRAIN_SUBSET` — the old code took the first 200 rows in
file order, so a smoke test measured whatever the data builder happened to
write first. And a **validation slice is held out** of the fit set, so the
training cell can report a loss on examples it did not train on. That slice
shares passages with the training set, so it detects overfitting rather than
estimating quality; the test split is the one that has no passages in common,
and it stays untouched until section 8.

In [ ]:
pool = list(train_data)
random.Random(SEED).shuffle(pool)

val_records = pool[:VAL_N] if VAL_N else []
rest = pool[VAL_N:]
fit_records = rest[:TRAIN_SUBSET] if TRAIN_SUBSET else rest


def as_turn_dict(records):
    prompts, responses = [], []
    for rec in records:
        q, a = format_io_as_turns(rec["input"], rec["output"])
        prompts.append(q)
        responses.append(a)
    return {"prompts": prompts, "responses": responses}


data = as_turn_dict(fit_records)
val_data = as_turn_dict(val_records) if val_records else None

print(f"fit {len(fit_records)}   val {len(val_records)}   "
      f"(of {len(train_data)} available)")
print("\nfit set by task:")
print(pd.Series([infer_task(r["input"]) for r in fit_records])
      .value_counts().to_string())
print("\n--- prompt ---\n", data["prompts"][0][:400])
print("\n--- response ---\n", data["responses"][0][:300])

In [ ]:
# Gemma closes a turn with <end_of_turn>, not <eos>. KerasHub's "auto" stop
# tokens only know about <eos>, so without this every generation runs all the
# way to max_length - that is most of the ten-to-twenty seconds per item, and
# it is why untuned output trails off into an invented second turn.
STOP_IDS = []
for _tok in (EOT, "<eos>"):
    try:
        _id = tokenizer.token_to_id(_tok)
        if _id is not None:
            STOP_IDS.append(int(_id))
    except Exception:
        pass
STOP_IDS = STOP_IDS or "auto"
print("stop token ids:", STOP_IDS)


def _strip(raw, prompt):
    """Keep the model's turn and nothing else."""
    out = raw[len(prompt):] if raw.startswith(prompt) else raw.replace(prompt, "")
    out = out.strip()
    for tok in (EOT, "<eos>", "<pad>"):
        i = out.find(tok)
        if i >= 0:
            out = out[:i].strip()
    for tok in (SOT + "user", SOT + "model", SOT):
        i = out.find(tok)
        if i > 0:
            out = out[:i].strip()
    return out


def _generate(prompts, max_length):
    return model.generate(prompts, max_length=max_length,
                          stop_token_ids=STOP_IDS)


def generate_answer(question, max_length=None):
    """One stateless generation. Returns the model text only."""
    p = build_prompt(question)
    return _strip(_generate(p, max_length or GEN_MAX_LENGTH), p)


def generate_many(questions, max_length=None, batch_size=None):
    """Batched generation - the difference between a seven-minute eval and a
    two-minute one. Falls back to one at a time if a batch runs out of memory.
    """
    bs = max(1, batch_size or GEN_BATCH_SIZE)
    ml = max_length or GEN_MAX_LENGTH
    prompts = [build_prompt(q) for q in questions]
    outs = []
    for i in range(0, len(prompts), bs):
        chunk = prompts[i:i + bs]
        if len(chunk) == 1:
            raws = [_generate(chunk[0], ml)]
        else:
            try:
                raws = list(_generate(chunk, ml))
            except Exception as exc:
                print(f"  batch of {len(chunk)} failed ({str(exc)[:90]}) - "
                      "falling back to one at a time")
                raws = [_generate(p, ml) for p in chunk]
        outs.extend(_strip(r, p) for r, p in zip(raws, chunk))
    return outs


def reset_generation():
    """Drop the cached, jit-compiled generate function.

    model.generate() compiles once and caches the result. enable_lora() adds
    variables and fit() changes them, so without this reset the "tuned"
    evaluation can be the untuned model answering - which looks like
    fine-tuning that did nothing at all.
    """
    try:
        model.generate_function = None
    except Exception as exc:
        print("could not reset the generate function:", exc)


reset_generation()
print("\nsmoke test of the generation path:")
print(generate_answer("[ANSWER] Answer using only the passage.\n\n"
                      "QUESTION: What is this place?\n\nPASSAGE:\n"
                      "Elmina Castle is a fortification on the coast of Ghana, "
                      "built by the Portuguese in 1482.")[:300])

## 7. Scoring

Defined once and run against both models, so the comparison is like for like.

| Task | What is measured |
| --- | --- |
| `classify` | JSON parses, then exact match per field and macro-F1 on theme |
| `answer` / `digest` / `summarize` | Grounding — every number and proper noun present in the passage — plus ROUGE-L against the reference |
| `refuse` | Did it refuse when it should have, **and** did it refuse when it should not have |

Grounding is an automatic proxy, not the 90% claim-support figure from the
proposal — that needs humans reading claims against passages. It tracks the
same failure cheaply enough to run after every change.

In [ ]:
RE_NUM = re.compile(r"\d[\d,.]*")
RE_PROPER = re.compile(r"\b[A-Z][a-zÀ-ɏ'’\-]{2,}\b")

# Verified against the gold refusals in the dataset: recall 1.00, zero false
# positives on gold answers. If you add refusal phrasings when you rebuild the
# data, re-check this regex against them or the metric drifts silently.
RE_REFUSAL = re.compile(
    r"((?:does|do)(?:\s+not|n't)\s+"
    r"(?:say|mention|cover|provide|describe|give|list|include|state|have)"
    r"|i (?:cannot|can't|am not able|do not have|don't have)"
    r"|cannot advise|can't advise|unable to|no information"
    r"|not covered|not mentioned|not stated"
    r"|source does not|passage does not)", re.I)

STOPWORD_PROPER = {
    "The", "This", "That", "It", "They", "There", "A", "An", "In", "On", "At",
    "If", "When", "What", "Which", "Where", "How", "You", "Your", "I", "We",
    "Check", "Passage", "Its", "His", "Her", "None", "True", "False",
}
GROUNDED_TASKS = ("answer", "digest", "summarize")


def norm_numbers(text):
    out = set()
    for n in RE_NUM.findall(text):
        n = n.rstrip(".,").replace(",", "")
        if n:
            out.add(n)
    return out


def extract_passage(inp):
    i = inp.find("PASSAGE:")
    return inp[i + len("PASSAGE:"):].strip() if i >= 0 else ""


def is_grounded(output, passage):
    '''True when every number and proper noun in the output is in the passage.'''
    if not passage or passage == "(none available)":
        return None
    if not norm_numbers(output) <= norm_numbers(passage):
        return False
    low = passage.lower()
    unseen = [w for w in RE_PROPER.findall(output)
              if w not in STOPWORD_PROPER and w.lower() not in low]
    return len(unseen) <= 1


def lcs_length(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(cur[j], prev[j + 1]))
        prev = cur
    return prev[-1]


def rouge_l(pred, ref):
    p, r = pred.lower().split(), ref.lower().split()
    if not p or not r:
        return 0.0
    l = lcs_length(p, r)
    if l == 0:
        return 0.0
    prec, rec = l / len(p), l / len(r)
    return 2 * prec * rec / (prec + rec)


def parse_labels(text):
    try:
        return json.loads(re.sub(r"^```(?:json)?|```$", "", text.strip(),
                                 flags=re.M).strip())
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except json.JSONDecodeError:
                return None
        return None


print("scoring helpers defined")

In [ ]:
def evaluate(records, label, n=None, verbose_examples=2):
    """Run the model over records and score per task. Returns (summary, rows)."""
    items = records[:n] if n else records
    t0 = time.time()

    # Generate first, in batches, then score. Scoring is pure Python and takes
    # no measurable time next to decoding.
    preds, step = [], max(1, GEN_BATCH_SIZE)
    for i in range(0, len(items), step):
        chunk = items[i:i + step]
        preds.extend(generate_many([r["input"] for r in chunk]))
        done = min(i + step, len(items))
        print(f"  [{label}] {done}/{len(items)}  ({time.time() - t0:.0f}s)")
    gen_seconds = time.time() - t0

    rows = []
    for rec, pred in zip(items, preds):
        task = rec.get("task") or infer_task(rec["input"])
        passage = extract_passage(rec["input"])
        row = {"task": task, "subtype": rec.get("subtype"),
               "country": rec.get("country"), "pred": pred, "ref": rec["output"]}
        if task == "classify":
            got, want = parse_labels(pred), parse_labels(rec["output"])
            row["json_ok"] = got is not None
            if got and want:
                for k in ("country", "theme", "terrain", "traveler_profile"):
                    row[f"hit_{k}"] = (str(got.get(k)).strip().lower()
                                       == str(want.get(k)).strip().lower())
                row["pred_theme"] = str(got.get("theme"))
                row["true_theme"] = str(want.get("theme"))
        elif task in GROUNDED_TASKS:
            row["grounded"] = is_grounded(pred, passage)
            row["rouge_l"] = rouge_l(pred, rec["output"])
            row["refused"] = bool(RE_REFUSAL.search(pred))
            # The gold answer being a refusal means this item is really a
            # refusal item, whatever the tag says.
            row["gold_refusal"] = bool(RE_REFUSAL.search(str(rec["output"])))
        elif task == "refuse":
            row["refused"] = bool(RE_REFUSAL.search(pred))
            row["gold_refusal"] = True
        rows.append(row)

    for rec, pred, row in list(zip(items, preds, rows))[:verbose_examples]:
        print(f"\n[{label}] {row['task']}")
        print("  prompt  :", rec["input"][:150].replace("\n", " "), "...")
        print("  expected:", str(rec["output"])[:140].replace("\n", " "))
        print("  got     :", pred[:140].replace("\n", " "))

    df = pd.DataFrame(rows)
    s = {"model": label, "n": len(df)}

    cl = df[df.task == "classify"] if "task" in df else df.iloc[0:0]
    if len(cl):
        s["classify_json_ok"] = round(cl.json_ok.mean(), 3)
        for k in ("country", "theme", "terrain", "traveler_profile"):
            col = f"hit_{k}"
            if col in cl:
                s[f"classify_{k}"] = round(cl[col].fillna(False).mean(), 3)
        ok = cl.dropna(subset=["pred_theme", "true_theme"]) \
            if "pred_theme" in cl else cl.iloc[0:0]
        if len(ok):
            try:
                from sklearn.metrics import f1_score
                s["classify_theme_macroF1"] = round(
                    f1_score(ok.true_theme, ok.pred_theme,
                             average="macro", zero_division=0), 3)
            except Exception:
                pass

    gen = df[df.task.isin(GROUNDED_TASKS)] if "task" in df else df.iloc[0:0]
    if len(gen):
        # .astype(bool) matters: the column holds Python bools and Nones, so
        # pandas infers object dtype and `~` on it raises.
        answerable = gen[~gen["gold_refusal"].fillna(False).astype(bool)] \
            if "gold_refusal" in gen else gen
        g = gen.grounded.dropna()
        s["grounded"] = round(g.mean(), 3) if len(g) else None
        s["rouge_l"] = round(gen.rouge_l.mean(), 3)
        if len(answerable):
            s["false_refusal"] = round(answerable.refused.mean(), 3)

    sm = df[df.task == "summarize"] if "task" in df else df.iloc[0:0]
    if len(sm):
        gs = sm.grounded.dropna()
        s["summarize_grounded"] = round(gs.mean(), 3) if len(gs) else None
        s["summarize_rouge_l"] = round(sm.rouge_l.mean(), 3)

    # Anything whose gold output is a refusal, however it was tagged.
    if "gold_refusal" in df:
        rf = df[df.gold_refusal.fillna(False).astype(bool)]
        if len(rf):
            s["refusal_recall"] = round(rf.refused.mean(), 3)

    s["empty_output_rate"] = round(float((df.pred.str.len() == 0).mean()), 3)
    s["seconds_per_item"] = round(gen_seconds / max(1, len(df)), 1)

    print(f"\n=== {label} ===")
    for k, v in s.items():
        print(f"  {k:26s} {v}")
    return s, df

## 8. Baseline — score the model before touching it

This is the cell capstones skip. Do not skip it.

It runs with the same template, the same greedy decoding and the same stop
tokens as the tuned run in section 10, which is the only way the two columns
can be compared. Generation is the slow part — batched, roughly 3–6 seconds
per item on a T4 — so 20 items is a couple of minutes. Raise `EVAL_N` to 100
or more for the final write-up.

In [ ]:
eval_records = test_full.to_dict("records")
random.Random(SEED).shuffle(eval_records)

reset_generation()
baseline_summary, baseline_rows = evaluate(eval_records, "base (no LoRA)", n=EVAL_N)
baseline_rows.to_csv(f"{DRIVE}/eval_baseline_{RUN_NAME}.csv", index=False)

## 9. LoRA fine-tuning

`enable_lora` freezes the backbone and adds the low-rank adapters, so only a
fraction of a percent of the weights move. Keep the epoch count low: 1B models
over a narrow dataset lose instruction-following and refusal behaviour before
they gain much else.

**On the optimizer.** The previous revision set
`model.optimizer.learning_rate` on whatever optimizer `from_preset` happened
to leave behind and trained with that. This compiles explicitly:

- `SparseCategoricalCrossentropy(from_logits=True)` — the preprocessor hands
  back integer labels and a sample weight that is zero on the prompt, so the
  loss lands on the response tokens only.
- `AdamW`, with weight decay excluded from `bias` and `scale` variables.
  Decaying a normalisation scale toward zero is not regularisation, it is
  damage.
- `gradient_accumulation_steps=4`, which buys an effective batch of 4 on a
  card that only fits one example at a time. Gradients at batch 1 are noisy
  enough to matter at 2e-4.
- `sampler="greedy"` again, because `compile` resets it.

In [ ]:
model.backbone.enable_lora(rank=LORA_RANK)
model.preprocessor.sequence_length = SEQUENCE_LENGTH

opt_kwargs = dict(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
if GRAD_ACCUM_STEPS and GRAD_ACCUM_STEPS > 1:
    opt_kwargs["gradient_accumulation_steps"] = GRAD_ACCUM_STEPS
try:
    optimizer = keras.optimizers.AdamW(**opt_kwargs)
except TypeError as exc:
    print("this Keras does not take", list(opt_kwargs), "-", exc)
    optimizer = keras.optimizers.AdamW(learning_rate=LEARNING_RATE,
                                       weight_decay=WEIGHT_DECAY)
# Decay belongs on the projection matrices, not on biases or norm scales.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
    sampler="greedy",
)
reset_generation()

trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
total = sum(int(np.prod(w.shape)) for w in model.weights)
print(f"LoRA rank {LORA_RANK}")
print(f"trainable {trainable:,} of {total:,}  ({100 * trainable / total:.2f}%)")
print("lr", float(model.optimizer.learning_rate),
      "| weight decay", WEIGHT_DECAY,
      "| effective batch", BATCH_SIZE * max(1, GRAD_ACCUM_STEPS))
assert trainable < total * 0.05, (
    "almost everything is trainable - enable_lora did not take effect, and "
    "this run will full-fine-tune a 1B model on a T4")

In [ ]:
PROBE = (
    "[ANSWER] Answer the question using only the passage below. "
    "If the passage does not contain the answer, say so plainly.\n\n"
    "QUESTION: How much does it cost to visit?\n\nPASSAGE:\n"
    "Kakum National Park, in the Central Region of Ghana, covers 375 square "
    "kilometres of tropical forest. It is one of only three places in Africa "
    "with a canopy walkway, 350 metres long and linking seven tree tops.")

class SaveLoRA(keras.callbacks.Callback):
    """Write the LoRA weights to Drive every CHECKPOINT_EVERY steps.

    A Colab disconnect two hours into a run otherwise loses everything. To
    resume after one, reload the latest checkpoint with the snippet in section
    12 and score it - no need to retrain from zero.
    """

    def __init__(self, every):
        super().__init__()
        self.every, self.step = every, 0

    def on_train_batch_end(self, batch, logs=None):
        self.step += 1
        if self.every and self.step % self.every == 0:
            ws = [w for w in self.model.weights
                  if "lora" in (getattr(w, "path", None) or w.name).lower()]
            path = f"{DRIVE}/ijeoma_lora_{RUN_NAME}_ckpt.npz"
            np.savez(path,
                     names=np.array([getattr(w, "path", None) or w.name for w in ws]),
                     step=self.step,
                     **{f"w{j}": np.asarray(w) for j, w in enumerate(ws)})
            print(f"\n  checkpoint at step {self.step} -> {path}")


ckpt_cb = SaveLoRA(CHECKPOINT_EVERY)
history = []
for epoch in range(NUM_EPOCHS):
    print(f"\n===== epoch {epoch + 1}/{NUM_EPOCHS} =====")
    h = model.fit(data, epochs=1, batch_size=BATCH_SIZE, shuffle=True, verbose=1,
                  callbacks=[ckpt_cb])
    entry = {k: float(v[-1]) for k, v in h.history.items()}

    if val_data:
        ev = model.evaluate(val_data, batch_size=BATCH_SIZE, verbose=0,
                            return_dict=True)
        entry.update({f"val_{k}": float(v) for k, v in ev.items()})
        print("held-out:", {k: round(float(v), 4) for k, v in ev.items()})

    history.append(entry)

    # The weights moved, so the cached generate function is stale.
    reset_generation()
    # The probe asks for a fee the passage does not give. The right answer is
    # to say so. If it starts inventing a price, stop and reconsider.
    print("\nprobe (correct answer = 'the passage does not say'):")
    print(" ", generate_answer(PROBE)[:300])

hist_df = pd.DataFrame(history)
print("\n", hist_df)

if "val_loss" in hist_df and len(hist_df) > 1:
    if hist_df["val_loss"].iloc[-1] > hist_df["val_loss"].min() + 1e-4:
        best = int(hist_df["val_loss"].idxmin()) + 1
        print(f"\nval_loss bottomed out at epoch {best} and rose after. That is "
              f"overfitting: set NUM_EPOCHS = {best} for the real run.")
if any(np.isnan(e.get("loss", 0.0)) for e in history):
    print("\nNaN loss. If MIXED_PRECISION is on, turn it off - float16 and "
          "LoRA on JAX do this. If it is already off, lower LEARNING_RATE.")

## 10. Score the fine-tuned model

In [ ]:
reset_generation()
tuned_summary, tuned_rows = evaluate(eval_records, f"LoRA {RUN_NAME}", n=EVAL_N)
tuned_rows.to_csv(f"{DRIVE}/eval_tuned_{RUN_NAME}.csv", index=False)

In [ ]:
comparison = pd.DataFrame([baseline_summary, tuned_summary]).set_index("model")
display(comparison.T)

metrics = [m for m in ("classify_json_ok", "classify_theme", "classify_country",
                       "grounded", "rouge_l", "summarize_grounded",
                       "refusal_recall")
           if m in comparison.columns and comparison[m].notna().all()]
if metrics:
    base_v = [comparison.iloc[0][m] for m in metrics]
    tuned_v = [comparison.iloc[1][m] for m in metrics]
    x = np.arange(len(metrics))
    plt.figure(figsize=(9, 3.6))
    plt.bar(x - 0.2, base_v, 0.4, label="before", color="#9ca3af")
    plt.bar(x + 0.2, tuned_v, 0.4, label="after LoRA", color="#2563eb")
    plt.xticks(x, metrics, rotation=20, ha="right")
    plt.ylim(0, 1)
    plt.ylabel("score")
    plt.title(f"Before vs after - {RUN_NAME}  (n={EVAL_N})")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("not enough shared metrics to plot - raise EVAL_N")

**Read `false_refusal` alongside `refusal_recall`.** A model that refuses
everything scores a perfect `refusal_recall` and is useless. If
`false_refusal` climbs after training, you have over-weighted the refusal
slice — cut it in the next data build rather than reaching for the learning
rate.

## 10b. Metrics promised in the proposal

The proposal commits to more than the core scores above: per-class precision,
recall and F1 for classification, ROUGE-1 and BERTScore for summaries and
answers, and an LLM-as-judge rubric (1-5 on factual accuracy, cultural nuance
and helpfulness).

**`claims_supported_rate` is the submitted proposal's success criterion:** the
share of factual claims in answers and summaries that the source passage
supports. Target 0.90. The judge splits each answer into claims and checks each
against the passage. This section adds them to both summaries, so they land in
the experiment log alongside everything else.

BERTScore runs on the CPU with `distilbert-base-uncased`: JAX has already
claimed the GPU, and the default `roberta-large` would be slow on CPU. Report
the model used next to the number.

In [ ]:
!pip install -q bert-score

from collections import Counter as _C
from sklearn.metrics import classification_report


def rouge_1(pred, ref):
    p, r = _C(pred.lower().split()), _C(ref.lower().split())
    overlap = sum((p & r).values())
    if not overlap:
        return 0.0
    prec, rec = overlap / sum(p.values()), overlap / sum(r.values())
    return 2 * prec * rec / (prec + rec)


def openai_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.environ.get("OPENAI_API_KEY")


JUDGE_PROMPT = """You are grading a travel assistant's answer about a West African destination.
Score it against the SOURCE PASSAGE only, from 1 (poor) to 5 (excellent), on:
- factual_accuracy: every claim is supported by the passage; nothing invented
- cultural_nuance: respectful, specific, free of stereotypes and Western travel cliches
- helpfulness: actually answers what was asked, clearly and concisely
Also list every atomic factual claim in the answer (one fact each: a date, a
builder, an event and a location are separate claims) and mark whether the
passage directly supports it. A refusal or a sentence with no factual content
contributes no claims.
Return JSON only: {{"factual_accuracy": n, "cultural_nuance": n, "helpfulness": n,
                    "claims": [{{"claim": "...", "supported": true}}]}}

QUESTION / TASK:
{task}

SOURCE PASSAGE:
{passage}

ASSISTANT ANSWER:
{answer}"""


def judge(task_text, passage, answer, key):
    import urllib.request
    body = json.dumps({
        "model": JUDGE_MODEL,
        "messages": [{"role": "user", "content": JUDGE_PROMPT.format(
            task=task_text, passage=passage[:6000], answer=answer)}],
        "response_format": {"type": "json_object"},
    }).encode()
    for attempt in range(3):
        try:
            req = urllib.request.Request(
                "https://api.openai.com/v1/chat/completions", data=body,
                headers={"Authorization": f"Bearer {key}",
                         "Content-Type": "application/json"})
            with urllib.request.urlopen(req, timeout=120) as r:
                return json.loads(json.load(r)["choices"][0]["message"]["content"])
        except Exception as exc:
            if attempt == 2:
                print("  judge failed:", exc)
            time.sleep(2 * (attempt + 1))
    return None


def extra_metrics(rows, summary, label):
    items = eval_records[:len(rows)]
    rows = rows.copy()
    rows["input"] = [r["input"] for r in items]

    # classification: per-class precision / recall / F1 on theme
    cl = rows.dropna(subset=["pred_theme", "true_theme"]) if "pred_theme" in rows else rows.iloc[0:0]
    if len(cl):
        rep = classification_report(cl.true_theme, cl.pred_theme,
                                    output_dict=True, zero_division=0)
        summary["classify_theme_macroP"] = round(rep["macro avg"]["precision"], 3)
        summary["classify_theme_macroR"] = round(rep["macro avg"]["recall"], 3)
        print(f"\n[{label}] theme classification report")
        print(classification_report(cl.true_theme, cl.pred_theme, zero_division=0))

    # text generation: ROUGE-1 and BERTScore on answerable grounded items
    gen = rows[rows.task.isin(GROUNDED_TASKS)]
    if "gold_refusal" in gen:
        gen = gen[~gen.gold_refusal.fillna(False).astype(bool)]
    if len(gen):
        summary["rouge_1"] = round(float(np.mean(
            [rouge_1(p, r) for p, r in zip(gen.pred, gen.ref)])), 3)
        try:
            from bert_score import score as bscore
            _, _, f1 = bscore(list(gen.pred), list(gen.ref), lang="en",
                              model_type="distilbert-base-uncased",
                              device="cpu", verbose=False)
            summary["bertscore_f1"] = round(float(f1.mean()), 3)
        except Exception as exc:
            print("BERTScore skipped:", exc)

        key = openai_key()
        if key:
            sample = gen.sample(min(JUDGE_N, len(gen)), random_state=SEED)
            scores = []
            for _, r in sample.iterrows():
                inp = r["input"]
                q = inp[:inp.find("PASSAGE:")].strip()
                s = judge(q, extract_passage(inp), r["pred"], key)
                if s:
                    claims = s.pop("claims", None) or []
                    s["claims_total"] = len(claims)
                    s["claims_supported"] = sum(bool(c.get("supported")) for c in claims)
                    scores.append(s)
            if scores:
                jd = pd.DataFrame(scores).apply(pd.to_numeric, errors="coerce")
                for k in ("factual_accuracy", "cultural_nuance", "helpfulness"):
                    if k in jd:
                        summary[f"judge_{k}"] = round(float(jd[k].mean()), 2)
                summary["judge_n"] = len(jd)
                # The proposal's success criterion: share of claims in answers
                # and summaries that the source passage supports (target 0.90).
                if {"claims_total", "claims_supported"} <= set(jd):
                    tot = jd["claims_total"].sum()
                    sup = jd[["claims_supported", "claims_total"]].min(axis=1).sum()
                    summary["claims_supported_rate"] = round(float(sup / tot), 3) if tot else None
        else:
            print("OPENAI_API_KEY not in Colab Secrets - LLM judge skipped")
    return summary


baseline_summary = extra_metrics(baseline_rows, baseline_summary, "base")
tuned_summary = extra_metrics(tuned_rows, tuned_summary, "tuned")

extra = [k for k in ("classify_theme_macroP", "classify_theme_macroR", "rouge_1",
                     "bertscore_f1", "judge_factual_accuracy",
                     "judge_cultural_nuance", "judge_helpfulness",
                     "claims_supported_rate", "judge_n")
         if k in baseline_summary or k in tuned_summary]
print("Success target (proposal): claims_supported_rate >= 0.90")
display(pd.DataFrame({"before": [baseline_summary.get(k) for k in extra],
                      "after LoRA": [tuned_summary.get(k) for k in extra]},
                     index=extra))


## 11. Invention rate

Prompts about places not in the corpus: invented names, real places far
outside West Africa, and real West African sites with no passage attached.
The correct behaviour in every case is to decline, because there is no source
to ground an answer in.

In [ ]:
INVENTED = ["Olumbe Rock Shelters", "Kanjiro Salt Flats", "Tembika Royal Baths",
            "Lake Wusuma", "Nzeribe Terraces", "Fort Adanwomase",
            "The Gidanwa Caves", "Mbakwe Sacred Pools", "Zarenga Cliff Tombs",
            "Opokun Heritage Village"]
ELSEWHERE = ["the Eiffel Tower", "Angkor Wat", "Machu Picchu", "Table Mountain",
             "the Pyramids of Giza", "Petra", "the Taj Mahal", "Uluru"]
REAL_NO_PASSAGE = ["Osun-Osogbo Sacred Grove", "Elmina Castle", "Sukur",
                   "the Bandiagara Escarpment", "Kakum National Park"]

prompts_adv = []
for n in INVENTED:
    prompts_adv.append((f"What is {n}?", "invented"))
    prompts_adv.append((f"Tell me what there is to see at {n}.", "invented"))
for n in ELSEWHERE:
    prompts_adv.append((f"What is {n}?", "outside_region"))
for n in REAL_NO_PASSAGE:
    prompts_adv.append((f"What is {n}?", "real_no_passage"))
random.Random(SEED).shuffle(prompts_adv)
prompts_adv = prompts_adv[:ADVERSARIAL_N]

TEMPLATE = ("[ANSWER] Answer the question using only the passage below. "
            "If the passage does not contain the answer, say so plainly.\n\n"
            "QUESTION: {q}\n\nPASSAGE:\n(none available)")

responses = generate_many([TEMPLATE.format(q=q) for q, _ in prompts_adv],
                          max_length=320)
adv = pd.DataFrame([
    {"kind": kind, "question": q, "response": out,
     "declined": bool(RE_REFUSAL.search(out))}
    for (q, kind), out in zip(prompts_adv, responses)
])
adv["invented"] = ~adv["declined"]

print("invention rate by kind (lower is better):")
print(adv.groupby("kind")["invented"].mean().round(3).to_string())
print(f"\noverall: {adv['invented'].mean():.1%}")
adv.to_csv(f"{DRIVE}/eval_adversarial_{RUN_NAME}.csv", index=False)

print("\n--- sample responses to invented places ---")
for _, r in adv[adv.kind == "invented"].head(4).iterrows():
    print(f"\n[{'INVENTED' if r.invented else 'declined'}] {r.question}\n  {r.response[:220]}")

## 12. Save the adapter and log the run

`model.save_weights` writes every weight in the model — 3.7 GB of base Gemma
you already have a copy of, to Drive, on every run. What is actually yours
after a LoRA run is the adapter: a few MB of low-rank matrices. That is what
gets saved here, with the reload recipe in a comment next to it. Set
`SAVE_FULL_WEIGHTS = True` in the config if you want the whole checkpoint as
well.

In [ ]:
def _weight_key(w):
    return (getattr(w, "path", None) or w.name)


lora_weights = [w for w in model.weights if "lora" in _weight_key(w).lower()]
if not lora_weights:
    raise RuntimeError("no LoRA weights found - did section 9 run?")

adapter_path = f"{DRIVE}/ijeoma_lora_{RUN_NAME}.npz"
names = [_weight_key(w) for w in lora_weights]
np.savez(adapter_path,
         names=np.array(names),
         **{f"w{i}": np.asarray(w) for i, w in enumerate(lora_weights)})
n_params = int(sum(np.prod(w.shape) for w in lora_weights))
print(f"saved {len(lora_weights)} LoRA tensors ({n_params:,} params) to")
print(" ", adapter_path)

# Reload it into a fresh session with:
#
#   model = kh.models.Gemma3CausalLM.from_preset(PRESET)
#   model.compile(sampler="greedy")
#   model.backbone.enable_lora(rank=LORA_RANK)
#   z = np.load(adapter_path)
#   saved = {str(n): z[f"w{i}"] for i, n in enumerate(z["names"])}
#   for w in model.weights:
#       key = getattr(w, "path", None) or w.name
#       if key in saved:
#           w.assign(saved[key])
#   model.generate_function = None

if SAVE_FULL_WEIGHTS:
    weights_path = f"{DRIVE}/ijeoma_full_{RUN_NAME}.weights.h5"
    model.save_weights(weights_path)
    print("saved full checkpoint (~3.7 GB):", weights_path)

In [ ]:
log_path = f"{DRIVE}/ijeoma_experiment_log.csv"
entry = {
    "timestamp": pd.Timestamp.now().isoformat(timespec="seconds"),
    "run_name": RUN_NAME,
    "preset": PRESET,
    "hf_secret": HF_SECRET_USED,
    "sampler": "greedy",
    "dataset": DATASET,
    "smoke_test": SMOKE_TEST,
    "n_train": len(fit_records),
    "n_val": len(val_records),
    "lora_rank": LORA_RANK,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "effective_batch": BATCH_SIZE * max(1, GRAD_ACCUM_STEPS),
    "sequence_length": SEQUENCE_LENGTH,
    "mixed_precision": MIXED_PRECISION,
    "final_loss": history[-1].get("loss") if history else None,
    "final_val_loss": history[-1].get("val_loss") if history else None,
    "eval_n": EVAL_N,
    "adapter_path": adapter_path if "adapter_path" in globals() else None,
    "invention_rate": (round(float(adv["invented"].mean()), 3)
                       if "adv" in globals() else None),
}
for k, v in tuned_summary.items():
    if k != "model":
        entry[f"tuned_{k}"] = v
for k, v in baseline_summary.items():
    if k != "model":
        entry[f"base_{k}"] = v

row = pd.DataFrame([entry])
if os.path.exists(log_path):
    row = pd.concat([pd.read_csv(log_path), row], ignore_index=True)
row.to_csv(log_path, index=False)
print("logged to", log_path)
display(row.tail(3))

## What to do with the numbers

**If the smoke test ran clean**, set `SMOKE_TEST = False`, raise `EVAL_N` to
100 or more, and do the three runs: rank 8 at 2e-4 for 2 epochs, rank 16 at
1e-4 for 3 epochs, then the better of the two on the full merged dataset.
Change one thing at a time; every run appends to
`ijeoma_experiment_log.csv`, which then becomes your results section.

**If the tuned column barely moved from the baseline**, check
`seconds_per_item` and `empty_output_rate` in both summaries first. A tuned
run that is suspiciously identical to the baseline usually means the cached
generate function was reused — `reset_generation()` exists for exactly that,
and it is called before each evaluation.

**If `grounded` did not improve**, the problem is the data, not the
hyperparameters. Look at which task is dragging — that is why scoring is
split by task.

**If `val_loss` rose while `loss` fell**, you are past the useful number of
epochs. The training cell prints the epoch where it bottomed out; use that.

**If `false_refusal` rose sharply**, there is too much refusal in the mix.
Rebuild with `--n-abstain` and `--n-scope` lowered. Read it next to
`refusal_recall`: a model that refuses everything scores a perfect recall and
is useless.

**If the invention rate is high on `real_no_passage`**, the model is reciting
from pretraining rather than reading the prompt. That is the failure the
grounded design exists to prevent, and it is worth a paragraph in your
write-up whichever way it goes.

**If the T4 runs out of memory**, in this order: lower `MAX_SEQUENCE_LENGTH`
only if the truncation warning stays quiet, drop `GEN_BATCH_SIZE` to 1, then
turn on `MIXED_PRECISION` (which on a Turing card means `mixed_float16`, and
watch for NaN loss).